In [0]:
%pip install google-api-python-client google-auth google-auth-httplib2 --quiet
%pip install openpyxl
%pip install pandas

In [0]:
dbutils.library.restartPython()

In [0]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, os, re, tempfile, shutil
from collections import defaultdict
from pyspark.sql import functions as F

# --- Configurações ---
FILE_ID = "1dJ89uPWhT_pZFcZom-T7c_DCVO3VjOWI3gAFDBy7n64"
FILE_NAME = "Histórico de Cotas.xlsx"
CATALOG = "hering-data-business-prod"
SCHEMA = "ext_industria"

# --- Credenciais Google ---
import json

# Credenciais armazenadas no Volume do Unity Catalog
VOLUME_CRED_PATH = "/Volumes/hering-data-business-prod/ext_industria/credentials/hering_data_engine_prod_b505839675d9.json"

credentials = service_account.Credentials.from_service_account_file(
    VOLUME_CRED_PATH,
    scopes=["https://www.googleapis.com/auth/drive"]
)

print("Configurações e credenciais carregadas com sucesso.")

In [0]:
from googleapiclient.discovery import build

drive_service = build("drive", "v3", credentials=credentials)

print("✅ Drive service criado com sucesso")

In [0]:
def download_file(file_id, file_name, dest_dir="/tmp"):
    """Baixa arquivo do Google Drive (suporta Google Sheets e arquivos normais)."""
    
    import os
    from googleapiclient.http import MediaIoBaseDownload

    os.makedirs(dest_dir, exist_ok=True)
    dest_path = os.path.join(dest_dir, file_name)

    # Descobrir tipo do arquivo
    file_metadata = drive_service.files().get(fileId=file_id, fields="mimeType").execute()
    mime_type = file_metadata["mimeType"]

    # Se for Google Sheets → exportar
    if mime_type == "application/vnd.google-apps.spreadsheet":
        request = drive_service.files().export_media(
            fileId=file_id,
            mimeType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
        )
        
        # garante extensão correta
        if not dest_path.endswith(".xlsx"):
            dest_path += ".xlsx"

    else:
        # arquivo normal (xlsx, csv, etc.)
        request = drive_service.files().get_media(fileId=file_id)

    # Download
    with open(dest_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()

    return dest_path


def read_file_to_df(file_path, extension, sheet_name=None, header_row=0):
    """
    Lê um arquivo local e retorna um Spark DataFrame.
    
    Args:
        file_path: Caminho do arquivo
        extension: Extensão do arquivo (.xlsx, .csv, etc.)
        sheet_name: Nome da aba (apenas para Excel). Se None, lê a primeira aba.
        header_row: Linha do cabeçalho (0-indexed). Use 1 para pular a primeira linha.
    """
    import pandas as pd
    
    ext = extension.lower()
    
    if ext in [".xlsx", ".xls"]:
        pdf = pd.read_excel(file_path, sheet_name=sheet_name, header=header_row, dtype=str)
        return spark.createDataFrame(pdf)
    elif ext == ".csv":
        return spark.read.option("header", "true").option("inferSchema", "false").csv(f"file:{file_path}")
    elif ext == ".parquet":
        return spark.read.parquet(f"file:{file_path}")
    elif ext == ".json":
        return spark.read.json(f"file:{file_path}")
    elif ext in [".txt", ".tsv"]:
        return spark.read.option("header", "true").option("delimiter", "\t").option("inferSchema", "false").csv(f"file:{file_path}")
    else:
        raise ValueError(f"Extensão não suportada: {ext}")

print("Funções de download e leitura definidas.")

In [0]:
# Processar arquivo com múltiplas abas e criar tabelas no Unity Catalog
tmp_dir = tempfile.mkdtemp(prefix="gdrive_")
error_files = []

# Configuração das abas a processar: (nome_aba, nome_tabela)
SHEETS_CONFIG = [
    ("Inclusão", "fato_cotas_inclusao"),
    ("Solicitação", "fato_cotas_solicitacao"),
    ("Pedidos PL", "fato_pedidos_PL"),
]

def sanitize_column_name(col_name):
    if col_name is None:
        return "coluna_sem_nome"
    sanitized = re.sub(r'[ ,;{}()\n\t=]+', '_', str(col_name))
    sanitized = sanitized.strip('_')
    sanitized = re.sub(r'_+', '_', sanitized)
    return sanitized if sanitized else "coluna_sem_nome"

def remove_blank_columns(df):
    """Remove colunas que são inteiramente nulas/vazias ou com nome 'Unnamed'."""
    cols_to_drop = []
    for col_name in df.columns:
        # Remover colunas com nome 'Unnamed' (geradas pelo pandas para colunas sem cabeçalho)
        if col_name.startswith("Unnamed") or col_name.startswith("coluna_sem_nome"):
            cols_to_drop.append(col_name)
            continue
        # Remover colunas onde todos os valores são nulos ou vazios
        non_null_count = df.filter(
            F.col(f"`{col_name}`").isNotNull() & (F.trim(F.col(f"`{col_name}`").cast("string")) != "")
        ).limit(1).count()
        if non_null_count == 0:
            cols_to_drop.append(col_name)
    return df.drop(*cols_to_drop), cols_to_drop

def auto_cast_columns(df, sample_size=1000):
    """Detecta e converte automaticamente os tipos das colunas (DATE, INTEGER, DOUBLE).
    Usa try_cast para tratar valores inválidos como NULL."""
    import re as _re
    
    # Padrões de data comuns
    date_patterns = [
        r'^\d{4}-\d{2}-\d{2}$',           # 2026-01-15
        r'^\d{2}/\d{2}/\d{4}$',           # 15/01/2026
        r'^\d{4}-\d{2}-\d{2}T',           # 2026-01-15T...
        r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}', # 2026-01-15 10:30
    ]
    
    casts_applied = []
    
    for col_name in df.columns:
        # Pegar amostra de valores não-nulos
        sample_rows = (
            df.filter(F.col(f"`{col_name}`").isNotNull() & (F.trim(F.col(f"`{col_name}`").cast("string")) != ""))
            .select(F.col(f"`{col_name}`").cast("string").alias("val"))
            .limit(sample_size)
            .collect()
        )
        
        if not sample_rows:
            continue
        
        values = [row["val"] for row in sample_rows if row["val"] is not None]
        if not values:
            continue
        
        # --- Tentar DATE ---
        is_date = False
        for pattern in date_patterns:
            matches = sum(1 for v in values if _re.match(pattern, v.strip()))
            if matches / len(values) >= 0.8:  # 80% dos valores batem
                is_date = True
                break
        
        if is_date:
            df = df.withColumn(col_name, F.to_date(F.col(f"`{col_name}`")))
            casts_applied.append((col_name, "DATE"))
            continue
        
        # --- Tentar INTEGER ---
        # Filtra valores que não são placeholders (como '-', 'N/A', etc.)
        numeric_values = [v.strip().replace(",", "") for v in values if v.strip() not in ("-", "N/A", "n/a", "")]
        
        if numeric_values:
            int_matches = 0
            for v in numeric_values:
                try:
                    num = float(v)
                    if num == int(num) and abs(num) < 2**31:
                        int_matches += 1
                except (ValueError, OverflowError):
                    pass
            
            if int_matches / len(numeric_values) >= 0.8:
                df = df.withColumn(col_name, F.expr(f"CAST(try_cast(`{col_name}` AS DOUBLE) AS INT)"))
                casts_applied.append((col_name, "INTEGER"))
                continue
            
            # --- Tentar DOUBLE ---
            double_matches = 0
            for v in numeric_values:
                try:
                    float(v)
                    double_matches += 1
                except ValueError:
                    pass
            
            if double_matches / len(numeric_values) >= 0.8:
                df = df.withColumn(col_name, F.expr(f"try_cast(`{col_name}` AS DOUBLE)"))
                casts_applied.append((col_name, "DOUBLE"))
                continue
    
    return df, casts_applied

try:
    print(f"\n{'='*60}")

    # Obter nome do arquivo
    file_metadata = drive_service.files().get(fileId=FILE_ID, fields="name").execute()
    FILE_NAME = file_metadata["name"]
    print(f"Arquivo: {FILE_NAME} (ID: {FILE_ID})")

    # Download do arquivo (uma única vez)
    print(f"  Baixando arquivo...")
    local_path = download_file(FILE_ID, FILE_NAME, tmp_dir)
    _, extension = os.path.splitext(local_path)
    print(f"  Arquivo baixado: {local_path}")

    # Processar cada aba configurada
    for sheet_name, table_name in SHEETS_CONFIG:
        print(f"\n{'-'*40}")
        print(f"Processando aba: '{sheet_name}'")
        
        try:
            # Ler aba com header na linha 2 (index 1) - pula a primeira linha
            df = read_file_to_df(local_path, extension, sheet_name=sheet_name, header_row=1)
            total_antes = df.count()

            # Remover linhas em branco (todas as colunas nulas ou vazias)
            condicao = F.lit(False)
            for col in df.columns:
                condicao = condicao | (F.col(f"`{col}`").isNotNull() & (F.trim(F.col(f"`{col}`")) != ""))
            df = df.filter(condicao)
            total_depois = df.count()

            print(f"  Linhas: {total_antes} → {total_depois} (removidas {total_antes - total_depois} linhas em branco)")
            print(f"  Colunas originais: {len(df.columns)}")

            # Sanitizar nomes das colunas
            for old_col in df.columns:
                new_col = sanitize_column_name(old_col)
                if new_col != old_col:
                    df = df.withColumnRenamed(old_col, new_col)

            # Remover colunas em branco
            df, dropped_cols = remove_blank_columns(df)
            if dropped_cols:
                print(f"  🗑️  Colunas removidas ({len(dropped_cols)}): {dropped_cols}")
            print(f"  Colunas restantes: {len(df.columns)}")

            # Ajuste automático de tipos
            df, casts = auto_cast_columns(df)
            if casts:
                for col_name, tipo in casts:
                    print(f"  🔄 Coluna '{col_name}' → {tipo}")
            else:
                print(f"  ℹ️  Nenhuma conversão de tipo aplicada")

            # Nome completo da tabela
            full_table_name = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"
            print(f"  Escrevendo tabela: {full_table_name} ({df.count()} linhas)")

            df.write.mode("overwrite") \
                .option("overwriteSchema", "true") \
                .saveAsTable(full_table_name)

            print(f"  ✅ Tabela criada com sucesso!")

        except Exception as e:
            print(f"  ❌ ERRO ao processar aba '{sheet_name}': {e}")
            error_files.append((sheet_name, str(e)))

    # Limpar arquivo local
    os.remove(local_path)

finally:
    shutil.rmtree(tmp_dir, ignore_errors=True)

print(f"\n{'='*60}")
if error_files:
    print(f"Processamento concluído com {len(error_files)} erro(s):")
    for name, err in error_files:
        print(f"  - {name}: {err}")
else:
    print(f"Processamento concluído com sucesso! {len(SHEETS_CONFIG)} tabelas criadas.")

In [0]:
%sql
-- teste github